# Lecture 12: Almost Complex Structures

**Source span.** Printed pages 65-71; physical PDF pages 79-85 in `Lectures on Symplectic Geometry.pdf` according to the course source map. I inspected the local PDF text for the almost-complex-structures span before revising this notebook.

**Lecture goal.** Compare symplectic, Riemannian, and complex linear structures, then make the compatibility condition `G_J(u,v)=Omega(u,Jv)` visible as matrix identities, positivity tests, and families of compatible almost complex structures.

The lecture's central triangle is algebraic: a symplectic form is skew and nondegenerate, a metric is symmetric positive, and an almost complex structure squares to `-Id`. Compatibility says these are not independent: `J` preserves `Omega`, `Omega(u,Ju)` is positive for nonzero `u`, and `g(u,v)=Omega(u,Jv)` becomes a Riemannian metric. The polar-decomposition proof shows why compatible structures always exist after choosing a metric.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, read_json, save_json, save_matplotlib
from utils.symplectic import standard_omega

ARTIFACT_TOPIC = "lecture-12"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
(ARTIFACT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "checks").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Artifact topic: {ARTIFACT_TOPIC}")

## Translation Guide And Library Routing

| Source idea | Computational object | Check |
| --- | --- | --- |
| almost complex structure | matrix `J` on a real vector space | `J @ J = -I` |
| `J squared equals minus identity` | residual norm `||J^2 + I||` | zero for a valid complex structure |
| compatibility with `Omega` | `J.T @ Omega @ J = Omega` and `G=Omega @ J` | `G` is symmetric positive definite |
| positive tame form | sampled values `u.T @ Omega @ J @ u` | all sampled values are positive for nonzero `u` |
| compatible metric | ellipses of `u.T @ G @ u = 1` | positivity and symmetry are visible |
| polar decomposition | factor a skew map into positive part times `J` | `J^2=-I` and `Omega(u,Jv)` positive |
| path-connected choices | a path `J_s=S_s J0 S_s^{-1}` | every sampled `s` remains compatible |
| homework group picture | `Sp(2n)`, `O(2n)`, `GL(n,C)`, and `U(n)` conditions | unitary samples satisfy all conditions |

**Library routing.** `numpy` is the right tool for finite-dimensional matrix identities, polar-decomposition residuals, eigenvalue positivity, and group-condition checks. `matplotlib` makes metric ellipses and matrix heatmaps inspectable. `networkx` is used only for the compatibility/polar proof dependency graph.

## Visualization Storyboard

1. **Compatibility matrix triangle.** Show the standard matrices `Omega0`, `J0`, and `g0`, with residuals for `J0^2=-I`, `J0` preserving `Omega0`, and `g0=Omega0 J0`.
2. **Compatible family and positive tame meter.** Vary a symplectic scaling `S_s` and inspect `J_s=S_s J0 S_s^{-1}`. The metric ellipse changes, but the compatibility residual stays small and the positivity floor stays positive.
3. **Polar decomposition and group intersections.** A proof graph records the construction `Omega + G -> A -> sqrt(AA*) -> J`, while a small matrix ledger checks that a unitary rotation lies in the intersections from the homework.

In [ ]:
# Standard compatibility triangle on R4 in (x1,x2,y1,y2) order.
Omega = standard_omega(2)
J0 = -Omega
I = np.eye(4)
G0 = Omega @ J0
J2_residual = float(np.linalg.norm(J0 @ J0 + I))
symplectic_residual = float(np.linalg.norm(J0.T @ Omega @ J0 - Omega))
metric_symmetry_residual = float(np.linalg.norm(G0 - G0.T))
metric_min_eig = float(np.min(np.linalg.eigvalsh(G0)))
assert J2_residual < 1e-12
assert symplectic_residual < 1e-12
assert metric_symmetry_residual < 1e-12
assert metric_min_eig > 0.99

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.7), gridspec_kw={"width_ratios": [1, 1, 1, 1.25]})
matrices = [(Omega, "Omega0"), (J0, "J0"), (G0, "g0=Omega0 J0")]
for ax, (matrix, title) in zip(axes[:3], matrices):
    im = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_xticks(range(4), labels=["x1", "x2", "y1", "y2"], rotation=45)
    ax.set_yticks(range(4), labels=["x1", "x2", "y1", "y2"])
    for (i, j), value in np.ndenumerate(matrix):
        if abs(value) > 1e-9:
            ax.text(j, i, f"{value:.0f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=axes[:3], fraction=0.025)
axes[3].axis("off")
ledger = [
    f"||J0^2 + I|| = {J2_residual:.1e}",
    f"||J0.T Omega J0 - Omega|| = {symplectic_residual:.1e}",
    f"||g0 - g0.T|| = {metric_symmetry_residual:.1e}",
    f"min eig(g0) = {metric_min_eig:.1f}",
    "Compatibility closes the triangle.",
]
axes[3].text(0.02, 0.95, "\n".join(ledger), va="top", fontsize=10, bbox={"boxstyle": "round,pad=0.45", "fc": "white", "ec": "0.55"})
fig.suptitle("Three geometries in one linear model: symplectic, metric, complex")
triangle_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "compatibility-matrix-triangle.png")
plt.close(fig)
display_artifact(triangle_path, width=760)
print({"J2_residual": J2_residual, "metric_min_eig": metric_min_eig})

In [ ]:
# A path of compatible complex structures from symplectic scalings.
def symplectic_scaling(s):
    scales = np.array([np.exp(s), np.exp(-0.5*s), np.exp(-s), np.exp(0.5*s)])
    return np.diag(scales)

def compatible_J(s):
    S = symplectic_scaling(s)
    return S @ J0 @ np.linalg.inv(S)

def metric_from_J(J):
    return Omega @ J

sample_s = np.linspace(-1.0, 1.0, 41)
compat_residuals = []
positivity_floors = []
for s in sample_s:
    J = compatible_J(s)
    G = metric_from_J(J)
    compat_residuals.append(float(np.linalg.norm(J.T @ Omega @ J - Omega) + np.linalg.norm(J @ J + I)))
    positivity_floors.append(float(np.min(np.linalg.eigvalsh(G))))
assert max(compat_residuals) < 1e-10
assert min(positivity_floors) > 0

theta = np.linspace(0, 2*np.pi, 240)
unit = np.vstack([np.cos(theta), np.sin(theta)])
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for s, color in [(-1.0, "#2c7fb8"), (0.0, "#1b9e77"), (1.0, "#d95f02")]:
    G2 = metric_from_J(compatible_J(s))[[0, 2]][:, [0, 2]]
    eigvals, eigvecs = np.linalg.eigh(G2)
    ellipse = eigvecs @ (unit / np.sqrt(eigvals[:, None]))
    axes[0].plot(ellipse[0], ellipse[1], color=color, lw=2, label=f"s={s:+.1f}")
axes[0].set_aspect("equal", adjustable="box")
axes[0].set_title("unit ellipses for g_s on one complex plane")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("y1")
axes[0].legend(fontsize=8)
axes[1].plot(sample_s, compat_residuals, color="#1b9e77", lw=2)
axes[1].set_yscale("log")
axes[1].set_title("compatibility residual along J_s")
axes[1].set_xlabel("s")
axes[1].set_ylabel("residual")
axes[2].plot(sample_s, positivity_floors, color="#d95f02", lw=2)
axes[2].set_title("positive tame floor: min eig(g_s)")
axes[2].set_xlabel("s")
axes[2].set_ylabel("minimum eigenvalue")
fig.suptitle("Path-connected compatible structures: the metric changes, positivity survives")
family_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "compatible-J-family-positive-metric-ellipses.png")
plt.close(fig)
display_artifact(family_path, width=760)
print({"max_compat_residual": max(compat_residuals), "min_positivity_floor": min(positivity_floors)})

In [ ]:
# Polar decomposition scaffold and homework-style group intersection checks.
A = J0  # with Euclidean G, Omega(u,v)=G(Au,v) in the lecture's convention v.T Omega u.
positive_part = np.eye(4)
J_polar = positive_part @ A
polar_J2_residual = float(np.linalg.norm(J_polar @ J_polar + I))
polar_compat_residual = float(np.linalg.norm(J_polar.T @ Omega @ J_polar - Omega))
assert polar_J2_residual < 1e-12
assert polar_compat_residual < 1e-12

phi = 0.37
X = np.array([[np.cos(phi), -np.sin(phi)], [np.sin(phi), np.cos(phi)]])
Y = np.array([[0.0, 0.0], [0.0, 0.0]])
U_real = np.block([[X, -Y], [Y, X]])
orthogonal_residual = float(np.linalg.norm(U_real.T @ U_real - I))
symplectic_group_residual = float(np.linalg.norm(U_real.T @ Omega @ U_real - Omega))
complex_commutator_residual = float(np.linalg.norm(U_real @ J0 - J0 @ U_real))
assert orthogonal_residual < 1e-12
assert symplectic_group_residual < 1e-12
assert complex_commutator_residual < 1e-12

G = nx.DiGraph()
G.add_edges_from([
    ("choose metric G", "identify Omega(u,.) with A"),
    ("identify Omega(u,.) with A", "A is skew-adjoint"),
    ("A is skew-adjoint", "AA* positive symmetric"),
    ("AA* positive symmetric", "sqrt(AA*)"),
    ("sqrt(AA*)", "J=(sqrt(AA*))^-1 A"),
    ("J=(sqrt(AA*))^-1 A", "J^2=-Id"),
    ("J=(sqrt(AA*))^-1 A", "Omega(Ju,Jv)=Omega(u,v)"),
    ("J=(sqrt(AA*))^-1 A", "Omega(u,Ju)>0"),
    ("symplectic basis", "explicit compatible J"),
    ("compatible J", "basis e_i, f_i=J e_i"),
])
pos = {
    "choose metric G": (0, 1.8),
    "identify Omega(u,.) with A": (1.7, 1.8),
    "A is skew-adjoint": (3.4, 1.8),
    "AA* positive symmetric": (5.1, 1.8),
    "sqrt(AA*)": (6.8, 1.8),
    "J=(sqrt(AA*))^-1 A": (8.5, 1.2),
    "J^2=-Id": (10.2, 2.2),
    "Omega(Ju,Jv)=Omega(u,v)": (10.2, 1.2),
    "Omega(u,Ju)>0": (10.2, 0.2),
    "symplectic basis": (5.1, -0.5),
    "explicit compatible J": (6.8, -0.5),
    "compatible J": (7.65, -1.1),
    "basis e_i, f_i=J e_i": (8.5, -0.5),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.3), gridspec_kw={"width_ratios": [1.55, 1]})
nx.draw_networkx_edges(G, pos, ax=axes[0], arrows=True, arrowstyle="-|>", arrowsize=13, edge_color="0.35")
nx.draw_networkx_nodes(G, pos, ax=axes[0], node_color="#c7e9c0", node_size=1800, edgecolors="0.25", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=axes[0], font_size=7)
axes[0].set_axis_off()
axes[0].set_title("polar-decomposition existence proof")
checks = [
    ("orthogonal", orthogonal_residual),
    ("symplectic", symplectic_group_residual),
    ("complex-linear", complex_commutator_residual),
]
axes[1].bar([name for name, _ in checks], [value for _, value in checks], color=["#2c7fb8", "#1b9e77", "#d95f02"])
axes[1].set_yscale("symlog", linthresh=1e-14)
axes[1].set_title("unitary sample lies in O ∩ Sp ∩ GL(C)")
axes[1].set_ylabel("condition residual")
for i, (_, value) in enumerate(checks):
    axes[1].text(i, max(value, 1e-15), f"{value:.1e}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Existence and homework ledger: polar J and the U(n) intersection")
polar_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "polar-decomposition-and-unitary-intersection-ledger.png")
plt.close(fig)
display_artifact(polar_path, width=760)
print({"polar_J2_residual": polar_J2_residual, "unitary_residuals": checks})

## Reading The Visuals

The matrix triangle is the standard model from the source span. `J0` rotates each `x_j` direction into `y_j` and each `y_j` direction back to `-x_j`, so `J0^2=-Id`. Multiplying `Omega0 @ J0` gives the Euclidean metric; the positive eigenvalues are the computational witness for the taming condition.

The family plot shows why compatible choices are flexible. A symplectic scaling changes the compatible complex structure and the metric ellipses, but the symplectic residual and positivity checks stay valid. This is the notebook's finite-dimensional stand-in for the path-connectedness result obtained from convex combinations of metrics and the polar construction.

The polar-decomposition ledger follows the proof: choose a metric, represent `Omega` by a skew-adjoint map `A`, split off a positive symmetric factor, and normalize the remaining factor into an almost complex structure. The homework group check records one unitary sample satisfying the orthogonal, symplectic, and complex-linear conditions at once.

In [ ]:
residuals = {
    "J_squared_plus_identity_residual": J2_residual,
    "J_preserves_omega_residual": symplectic_residual,
    "compatible_metric_symmetry_residual": metric_symmetry_residual,
    "compatible_metric_min_eigenvalue": metric_min_eig,
    "family_max_compatibility_residual": max(compat_residuals),
    "family_min_positivity_floor": min(positivity_floors),
    "polar_J_squared_residual": polar_J2_residual,
    "polar_compatibility_residual": polar_compat_residual,
    "unitary_orthogonal_residual": orthogonal_residual,
    "unitary_symplectic_residual": symplectic_group_residual,
    "unitary_complex_linear_residual": complex_commutator_residual,
}
residual_path = save_json(residuals, ARTIFACT_TOPIC, "checks", "almost-complex-residuals.json")

source_span = {
    "title": "Almost Complex Structures",
    "printed_pages": "65-71",
    "physical_pdf_pages": "79-85",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "sections": [
        "Three geometries",
        "Complex structures on vector spaces",
        "Compatible structures",
        "Homework 8 compatible linear structures themes",
    ],
    "concepts": [
        "almost complex structure",
        "J squared equals minus identity",
        "compatible metric",
        "positive tame form",
        "polar decomposition",
        "path-connected compatible choices",
    ],
    "copyright_note": "Source pages were used for terminology and coverage; notebook prose, visuals, and code are original and no page crops or long exercise text are copied.",
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Make compatibility of omega, J, and g visible through matrix identities, positivity ellipses, and polar-decomposition checks.",
    "source_span_read": source_span,
    "library_routing": [
        {"concept": "almost complex structure", "representation": "matrix heatmap and J^2 residual", "library": "numpy + matplotlib", "why": "the definition is a finite matrix identity"},
        {"concept": "compatible metric", "representation": "metric heatmap and eigenvalue checks", "library": "numpy + matplotlib", "why": "positivity is best shown by eigenvalues and ellipses"},
        {"concept": "positive tame form", "representation": "positivity floor along a compatible family", "library": "numpy + matplotlib", "why": "the taming condition is numerical positivity for nonzero vectors"},
        {"concept": "polar decomposition", "representation": "proof dependency graph", "library": "networkx", "why": "the existence proof is a sequence of linear-algebra implications"},
    ],
    "visual_sequence": [
        {"concept": "compatibility matrix triangle", "artifact": "artifacts/lecture-12/figures/compatibility-matrix-triangle.png", "inspection_target": "Omega, J, and g close the standard compatibility identities"},
        {"concept": "compatible family and positive metric", "artifact": "artifacts/lecture-12/figures/compatible-J-family-positive-metric-ellipses.png", "inspection_target": "metric ellipses change while compatibility and positivity persist"},
        {"concept": "polar decomposition and U(n) intersection", "artifact": "artifacts/lecture-12/figures/polar-decomposition-and-unitary-intersection-ledger.png", "inspection_target": "polar proof steps and group residuals support the linear-structure homework"},
    ],
    "computational_checks": residuals,
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")

final_sanity = {
    "artifacts": [
        str(triangle_path.relative_to(BOOK_ROOT)),
        str(family_path.relative_to(BOOK_ROOT)),
        str(polar_path.relative_to(BOOK_ROOT)),
        str(residual_path.relative_to(BOOK_ROOT)),
        str(source_path.relative_to(BOOK_ROOT)),
        str(storyboard_path.relative_to(BOOK_ROOT)),
    ],
    "assertions": {
        "J_squared_equals_minus_identity": bool(J2_residual < 1e-12),
        "J_preserves_omega": bool(symplectic_residual < 1e-12),
        "compatible_metric_positive": bool(metric_min_eig > 0.99),
        "compatible_family_stays_valid": bool(max(compat_residuals) < 1e-10),
        "positive_tame_floor_stays_positive": bool(min(positivity_floors) > 0),
        "polar_decomposition_recovers_complex_structure": bool(polar_J2_residual < 1e-12),
        "unitary_sample_satisfies_three_groups": bool(max(orthogonal_residual, symplectic_group_residual, complex_commutator_residual) < 1e-12),
    },
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print(json.dumps(final_sanity, indent=2))

In [ ]:
final_sanity = read_json(ARTIFACT_ROOT / "checks" / "final-sanity.json")
assert all(final_sanity["assertions"].values())
for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    assert path.exists(), relative
    assert path.stat().st_size > 100, relative

for relative in final_sanity["artifacts"]:
    path = BOOK_ROOT / relative
    if path.suffix.lower() in {".png", ".html", ".svg"}:
        display_artifact(path, width=760, height=430 if path.suffix == ".html" else None)

print("Lecture 12 checks passed; artifacts are present and nonempty.")

## Takeaways

- An almost complex structure is a smooth field of linear maps with `J^2=-Id`.
- Compatibility with a symplectic form means `J` preserves the form and `Omega(u,Ju)` is positive for every nonzero vector.
- The compatible metric is `g(u,v)=Omega(u,Jv)`; symmetry and positive eigenvalues are the checks to keep nearby.
- Polar decomposition gives compatible almost complex structures from any symplectic form after choosing a Riemannian metric.
- Compatible structures are flexible: the lecture's path-connectedness result is reflected in the sampled compatible family where residuals stay zero and positivity stays positive.